# Polyak 和 Nesterov 的 long-time loss curve

这里不固定 data budget $D=BK$，而是直接延长训练时间，观察 loss 从 signal-dominated decay 进入 noise-dominated plateau：

- **实线**：数值模拟；
- **虚线**：scaling-law prediction。

Polyak 和 Nesterov 分别画在两个 panel 中。Notebook 完全独立，只依赖 NumPy、PyTorch、Matplotlib 和 tqdm。

In [ ]:
from dataclasses import dataclass
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
from tqdm.auto import tqdm

# ===== Long-time experiment =====
SEED = 0
DEVICE = "auto"       # "auto", "cpu", "cuda", "cuda:0", or "mps"

BETA = 4.0
S = 0.3
SIGMA = 0.5
DIM = 4096
RUNS = 100
TRAIN_STEPS = 500_000
SHOW_PROGRESS = True

# 这组参数让 crossover 落在可观察的训练区间内：
# Polyak: transient -> scaling -> noise
# Nesterov: scaling -> noise；其 transient 被理论 prefactor 强烈压低，不会主导。
SETTINGS = {
    "polyak":   {"B": 128, "eta": 0.01, "rho": 0.999},
    "nesterov": {"B": 4,   "eta": 0.01, "rho": 0.999},
}


## 模型和 simulation

使用原 setting 的 power-law spectrum、target smoothness、label noise 和 curvature-aligned Gaussian gradient noise：

$$
\lambda_j=j^{-\beta},\qquad
\lambda_j|\theta_j^*|^2=j^{-(1+s\beta)}.
$$

每个算法运行固定的 `TRAIN_STEPS`，所以总数据量不再被预先固定。

In [ ]:
def get_device(name):
    if name != "auto":
        return torch.device(name)
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


@dataclass
class Problem:
    lam: torch.Tensor
    theta_star: torch.Tensor
    sigma: float


def build_problem(dim, beta, s, sigma, device, dtype):
    j = torch.arange(1, dim + 1, device=device, dtype=dtype)
    lam = j.pow(-beta)
    theta_star = j.pow(-0.5 * (1.0 + (s - 1.0) * beta))
    return Problem(lam, theta_star, sigma)


def loss(delta, lam):
    return 0.5 * torch.sum(lam * delta.square(), dim=-1)


@torch.no_grad()
def simulate(problem, algorithm, B, eta, rho, steps, runs, seed):
    generator = torch.Generator(device=problem.lam.device)
    generator.manual_seed(seed)

    delta = -problem.theta_star.expand(runs, -1).clone()
    previous = delta.clone()

    # 前 100 步密集记录，之后 logarithmic sampling。
    checkpoints = set(range(min(steps, 100) + 1))
    checkpoints.update(np.geomspace(1, steps, 500).round().astype(int).tolist())
    checkpoints.add(steps)

    saved_steps = [0]
    saved_losses = [loss(delta, problem.lam).mean().item()]

    iterator = tqdm(
        range(1, steps + 1),
        total=steps,
        desc=f"{algorithm.capitalize()} training",
        unit="step",
        mininterval=0.5,
        disable=not SHOW_PROGRESS,
    )

    for k in iterator:
        momentum = delta - previous
        evaluation = delta + rho * momentum if algorithm == "nesterov" else delta

        evaluation_loss = loss(evaluation, problem.lam)
        scale = ((evaluation_loss + problem.sigma**2) / B).sqrt().unsqueeze(-1)
        noise = torch.randn(
            evaluation.shape, device=evaluation.device,
            dtype=evaluation.dtype, generator=generator,
        )
        gradient = evaluation * problem.lam + noise * problem.lam.sqrt() * scale

        new_delta = delta + rho * momentum - eta * gradient
        previous, delta = delta, new_delta

        if k in checkpoints:
            saved_steps.append(k)
            saved_losses.append(loss(delta, problem.lam).mean().item())

    return np.asarray(saved_steps), np.asarray(saved_losses)

## Scaling-law prediction 和有效转折点

定义 $\eta_\rho=\eta/(1-\rho)$。理论只给出 $\asymp$：

$$
L_{\rm Polyak}(k)
\asymp
\rho^k+(\eta_\rho k)^{-s}+rac{\sigma^2}{B}\eta_\rho,
$$

$$
L_{\rm Nesterov}(k)
\asymp
\left(\frac{(1-\rho)^2}{\eta}\right)^{s+1/\beta}
\left(\frac{4\rho^2}{(1+\rho)^2}\right)^k
+(\eta_\rho k)^{-s}
+rac{\sigma^2}{B}\min\{\eta_\rho,\eta_\rho^{1/\beta}\}.
$$

因此数值预测写成

$$
L_{\rm pred}(k)=C_T T(k)+C_S S(k)+C_N N(k),
\qquad C_T,C_S,C_N\ge0.
$$

三个 $O(1)$ 系数用 simulation curve 做非负 relative least-squares 拟合。它们只用于恢复 $\asymp$ 中没有给出的常数；最终图仍然只画一条总 prediction 虚线，不展示 component curves。

有效的 scaling-to-noise balance point 由拟合后的两项相等定义：

$$
C_S(\eta_\rho k_*)^{-s}=C_NN.
$$

这比假设 $C_S=C_N=1$ 得到的 unit-coefficient scale 更可靠。由于两项之和是光滑的，$k_*$ 是 balance scale，不是尖锐折点。


In [ ]:
def theory_matrix(steps, algorithm, B, eta, rho):
    k = np.maximum(np.asarray(steps, dtype=float), 1.0)
    eta_rho = eta / (1.0 - rho)
    signal = (eta_rho * k) ** (-S)

    if algorithm == "polyak":
        transient = rho**k
        noise = np.full_like(k, SIGMA**2 * eta_rho / B)
    else:
        r_nes = 4.0 * rho**2 / (1.0 + rho)**2
        transient = ((1.0 - rho)**2 / eta) ** (S + 1.0 / BETA) * r_nes**k
        floor = SIGMA**2 / B * min(eta_rho, eta_rho ** (1.0 / BETA))
        noise = np.full_like(k, floor)

    return np.column_stack([transient, signal, noise])


def nonnegative_lstsq(X, y):
    # Only three variables: enumerate all nonempty active sets, avoiding SciPy.
    best_coef = np.zeros(X.shape[1])
    best_error = np.inf
    for mask in range(1, 1 << X.shape[1]):
        active = [j for j in range(X.shape[1]) if mask & (1 << j)]
        coef_active, *_ = np.linalg.lstsq(X[:, active], y, rcond=None)
        if np.any(coef_active < 0):
            continue
        coef = np.zeros(X.shape[1])
        coef[active] = coef_active
        error = np.mean((X @ coef - y) ** 2)
        if error < best_error:
            best_coef, best_error = coef, error
    return best_coef


def fit_prediction(steps, empirical, algorithm, B, eta, rho):
    X = theory_matrix(steps, algorithm, B, eta, rho)
    eta_rho = eta / (1.0 - rho)
    valid = (steps > 0) & (eta_rho * steps >= 1.0)

    # Relative least squares: each log-time region has comparable influence.
    scale = np.maximum(empirical[valid], np.finfo(float).tiny)
    coefficients = nonnegative_lstsq(X[valid] / scale[:, None], np.ones(valid.sum()))
    prediction = X @ coefficients
    return prediction, valid, coefficients


def fitted_transition_predictions(algorithm, B, eta, rho, coefficients):
    c_transient, c_signal, c_noise = coefficients
    eta_rho = eta / (1.0 - rho)
    X_one = theory_matrix(np.array([1.0]), algorithm, B, eta, rho)[0]
    noise_shape = X_one[2]

    # Fitted signal/noise equality. None means the fitted noise coefficient vanished.
    if c_signal > 0 and c_noise > 0 and noise_shape > 0:
        scaling_to_noise = (c_signal / (c_noise * noise_shape)) ** (1.0 / S) / eta_rho
    else:
        scaling_to_noise = None

    # Diagnose whether the fitted transient ever dominates the fitted signal.
    if c_transient <= 0 or c_signal <= 0:
        transient_to_scaling = None
        max_ratio = 0.0
    else:
        if algorithm == "polyak":
            decay_base = rho
            prefactor = 1.0
        else:
            decay_base = 4.0 * rho**2 / (1.0 + rho)**2
            prefactor = ((1.0 - rho)**2 / eta) ** (S + 1.0 / BETA)

        decay_rate = -np.log(decay_base)
        k_at_max = max(1.0, S / decay_rate)
        max_ratio = (
            c_transient * prefactor * decay_base**k_at_max
            / (c_signal * (eta_rho * k_at_max) ** (-S))
        )

        # Search for the last crossing of fitted transient and fitted signal.
        grid = np.geomspace(1.0, max(float(TRAIN_STEPS), 10.0), 20_000)
        log_ratio = (
            np.log(c_transient * prefactor / c_signal)
            + grid * np.log(decay_base)
            + S * np.log(eta_rho * grid)
        )
        crossings = np.flatnonzero(log_ratio[:-1] * log_ratio[1:] < 0)
        if crossings.size:
            i = crossings[-1]
            left, right = grid[i], grid[i + 1]
            for _ in range(80):
                middle = 0.5 * (left + right)
                value = (
                    np.log(c_transient * prefactor / c_signal)
                    + middle * np.log(decay_base)
                    + S * np.log(eta_rho * middle)
                )
                left_value = (
                    np.log(c_transient * prefactor / c_signal)
                    + left * np.log(decay_base)
                    + S * np.log(eta_rho * left)
                )
                if left_value * value <= 0:
                    right = middle
                else:
                    left = middle
            transient_to_scaling = 0.5 * (left + right)
        else:
            transient_to_scaling = None

    return {
        "transient_to_scaling": transient_to_scaling,
        "scaling_to_noise": scaling_to_noise,
        "max_transient_signal_ratio": max_ratio,
    }


## 运行并画图

曲线后半段逐渐变平的位置就是 label noise 主导的区域。

In [ ]:
seed_everything(SEED)
device = get_device(DEVICE)
dtype = torch.float32 if device.type == "mps" else torch.float64
problem = build_problem(DIM, BETA, S, SIGMA, device, dtype)

results = {}
start = time.time()

for index, algorithm in enumerate(("polyak", "nesterov")):
    q = SETTINGS[algorithm]
    t0 = time.time()
    steps, empirical = simulate(
        problem, algorithm, q["B"], q["eta"], q["rho"],
        TRAIN_STEPS, RUNS, SEED + index * 10_000,
    )
    prediction, valid, coefficients = fit_prediction(
        steps, empirical, algorithm, q["B"], q["eta"], q["rho"]
    )
    transitions = fitted_transition_predictions(
        algorithm, q["B"], q["eta"], q["rho"], coefficients
    )
    results[algorithm] = (steps, empirical, prediction, valid, transitions)

    print(f"\n{algorithm.upper()}")
    print(f"B={q['B']}, eta={q['eta']}, rho={q['rho']}, steps={TRAIN_STEPS:,}")
    print(
        "fitted constants [transient, scaling, noise] = "
        + np.array2string(coefficients, precision=4)
    )
    print(f"max fitted transient/scaling ratio = {transitions['max_transient_signal_ratio']:.4g}")
    if transitions["transient_to_scaling"] is None:
        print("transient -> scaling: no fitted dominant-region crossing")
    else:
        print(f"transient -> scaling balance: k ≈ {transitions['transient_to_scaling']:,.0f}")
    if transitions["scaling_to_noise"] is None:
        print("scaling -> noise: fitted noise coefficient is zero")
    else:
        print(f"scaling -> noise balance:     k ≈ {transitions['scaling_to_noise']:,.0f}")
    print(f"simulation time: {time.time()-t0:.1f}s")

print(f"\ntotal time: {time.time()-start:.1f}s")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))

for ax, algorithm in zip(axes, ("polyak", "nesterov")):
    steps, empirical, prediction, valid, transitions = results[algorithm]
    positive = steps > 0

    ax.plot(steps[positive], empirical[positive], lw=2.2, label="Simulation")
    ax.plot(steps[valid], prediction[valid], "--", lw=2.2, label="Theory prediction")

    k_ts = transitions["transient_to_scaling"]
    if k_ts is not None and 1 <= k_ts <= TRAIN_STEPS:
        ax.axvline(
            k_ts, color="tab:green", ls=":", lw=1.6,
            label=rf"fitted $T=S$: {k_ts:,.0f}"
        )

    k_sn = transitions["scaling_to_noise"]
    if k_sn is not None and 1 <= k_sn <= TRAIN_STEPS:
        ax.axvline(
            k_sn, color="tab:red", ls=":", lw=1.6,
            label=rf"fitted $S=N$: {k_sn:,.0f}"
        )

    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel("Optimization step $k$")
    ax.set_ylabel("Population excess risk")
    ax.set_title(algorithm.capitalize())
    ax.grid(alpha=0.2)
    ax.legend(fontsize=8)

fig.suptitle(f"Long-time loss curves ({TRAIN_STEPS:,} steps)", y=1.02)
fig.tight_layout()
plt.show()


# Blow-up experiment：$\rho_t=1-1/t$

下面使用相同的 numerical parameters 比较 Polyak 和 Nesterov 的 blow-up time：

$$
\rho_t=1-\frac1t,
\qquad t=1,2,\ldots
$$

两种算法使用相同的 $B$、$\eta$、初始化、replica 数量和随机 seed。定义 blow-up time 为平均 population excess risk 首次超过 `BLOWUP_THRESHOLD` 的迭代；如果 loss 先变成非有限值，也立即判定为 blow-up。

由 stability scaling，可以先得到参考时间尺度：

$$
\text{Polyak:}\quad 1-\rho_t\asymp\frac{\eta}{B}
\quad\Longrightarrow\quad
 t_{\rm stab}^{\rm P}\asymp\frac{B}{\eta},
$$

$$
\text{Nesterov:}\quad 1-\rho_t\asymp\frac{\eta}{B^\beta}
\quad\Longrightarrow\quad
 t_{\rm stab}^{\rm N}\asymp\frac{B^\beta}{\eta}.
$$

这些是进入理论不稳定区域的 scaling-level 时间，不等于 loss 立即超过数值 threshold 的时间。实际 blow-up 通常会有一段延迟。

In [ ]:
# 两种算法完全相同的参数。
BLOWUP_B = 8
BLOWUP_ETA = 0.1
BLOWUP_MAX_STEPS = 200_000
BLOWUP_RUNS = RUNS
BLOWUP_THRESHOLD = 1e6

POLYAK_STABILITY_TIME = BLOWUP_B / BLOWUP_ETA
NESTEROV_STABILITY_TIME = BLOWUP_B**BETA / BLOWUP_ETA

print(f"Polyak predicted instability scale:   t ~ {POLYAK_STABILITY_TIME:,.0f}")
print(f"Nesterov predicted instability scale: t ~ {NESTEROV_STABILITY_TIME:,.0f}")
print(f"Numerical blow-up threshold: loss > {BLOWUP_THRESHOLD:g}")

## Dynamic-momentum simulation

在第 $t$ 次更新中直接使用 $\rho_t=1-1/t$。为公平比较，Polyak 和 Nesterov 的 generator 使用相同 seed。函数返回 loss curve 和首次达到 blow-up criterion 的时间。

In [ ]:
@torch.no_grad()
def simulate_blowup(
    problem, algorithm, *, B, eta, max_steps, runs, seed, threshold
):
    generator = torch.Generator(device=problem.lam.device)
    generator.manual_seed(seed)

    delta = -problem.theta_star.expand(runs, -1).clone()
    previous = delta.clone()

    # Dense recording near the expected Polyak instability time and log spacing later.
    checkpoints = set(range(0, min(max_steps, 1_000) + 1))
    checkpoints.update(np.geomspace(1, max_steps, 700).round().astype(int).tolist())
    checkpoints.add(max_steps)

    saved_steps = [0]
    saved_losses = [loss(delta, problem.lam).mean().item()]
    blowup_time = None

    iterator = tqdm(
        range(1, max_steps + 1),
        total=max_steps,
        desc=f"{algorithm.capitalize()} blow-up",
        unit="step",
        mininterval=0.5,
        disable=not SHOW_PROGRESS,
    )

    for t in iterator:
        rho_t = 1.0 - 1.0 / t
        momentum = delta - previous
        evaluation = delta + rho_t * momentum if algorithm == "nesterov" else delta

        evaluation_loss = loss(evaluation, problem.lam)
        scale = ((evaluation_loss + problem.sigma**2) / B).sqrt().unsqueeze(-1)
        noise = torch.randn(
            evaluation.shape, device=evaluation.device,
            dtype=evaluation.dtype, generator=generator,
        )
        gradient = evaluation * problem.lam + noise * problem.lam.sqrt() * scale

        new_delta = delta + rho_t * momentum - eta * gradient
        previous, delta = delta, new_delta
        mean_loss = loss(delta, problem.lam).mean()

        finite = bool(torch.isfinite(mean_loss).item())
        value = float(mean_loss.item()) if finite else math.inf

        if t in checkpoints or not finite or value > threshold:
            saved_steps.append(t)
            saved_losses.append(value)

        if not finite or value > threshold:
            blowup_time = t
            iterator.set_postfix(loss=f"{value:.2e}", blowup=t)
            break

        if t % 500 == 0:
            iterator.set_postfix(loss=f"{value:.2e}", rho=f"{rho_t:.6f}")

    return np.asarray(saved_steps), np.asarray(saved_losses), blowup_time

## 运行并比较 blow-up time

图中的点虚线表示理论 stability scale，叉号表示按统一 threshold 测得的 empirical blow-up time。若算法在 `BLOWUP_MAX_STEPS` 内没有 blow up，则报告为 censored observation。

In [ ]:
blowup_results = {}
blowup_start = time.time()

for algorithm in ("polyak", "nesterov"):
    t0 = time.time()
    steps_b, losses_b, blowup_time = simulate_blowup(
        problem,
        algorithm,
        B=BLOWUP_B,
        eta=BLOWUP_ETA,
        max_steps=BLOWUP_MAX_STEPS,
        runs=BLOWUP_RUNS,
        seed=SEED + 123_456,  # identical seed for both algorithms
        threshold=BLOWUP_THRESHOLD,
    )
    blowup_results[algorithm] = (steps_b, losses_b, blowup_time)

    if blowup_time is None:
        message = f"no blow-up by t={BLOWUP_MAX_STEPS:,}"
    else:
        message = f"blow-up at t={blowup_time:,}"
    print(f"{algorithm:9s}: {message}; runtime={time.time()-t0:.1f}s")

print(f"total blow-up experiment time: {time.time()-blowup_start:.1f}s")

fig, ax = plt.subplots(figsize=(8, 5.2))
colors = {"polyak": "tab:orange", "nesterov": "tab:blue"}

for algorithm in ("polyak", "nesterov"):
    steps_b, losses_b, blowup_time = blowup_results[algorithm]
    positive = steps_b > 0
    ax.plot(
        steps_b[positive], losses_b[positive],
        color=colors[algorithm], lw=2.0, label=algorithm.capitalize(),
    )
    if blowup_time is not None:
        ax.scatter(
            [blowup_time], [losses_b[-1]], marker="x", s=90,
            linewidths=2.2, color=colors[algorithm], zorder=5,
            label=f"{algorithm.capitalize()} blow-up: {blowup_time:,}",
        )

ax.axvline(
    POLYAK_STABILITY_TIME, color="tab:orange", ls=":", lw=1.8,
    label=rf"Polyak stability scale $B/\eta={POLYAK_STABILITY_TIME:,.0f}$",
)
ax.axvline(
    NESTEROV_STABILITY_TIME, color="tab:blue", ls=":", lw=1.8,
    label=rf"Nesterov stability scale $B^\beta/\eta={NESTEROV_STABILITY_TIME:,.0f}$",
)
ax.axhline(
    BLOWUP_THRESHOLD, color="black", ls="--", lw=1.3,
    label=f"blow-up threshold = {BLOWUP_THRESHOLD:g}",
)

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Iteration $t$")
ax.set_ylabel("Population excess risk")
ax.set_title(r"Dynamic momentum: $\rho_t=1-1/t$")
ax.grid(alpha=0.2)
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()